# Experiment: LargeST GLA Feasibility

This read-only spike checks whether switching from METR-LA to the LargeST Greater Los Angeles (GLA) subset is practical for this project.

Notebook goals:
- detect a local LargeST-GLA layout if one exists
- inspect raw flow, processed tensors, metadata, and graph files when available
- print shapes, timestamp coverage, node counts, and edge counts
- compare METR-LA with LargeST-GLA at a high level


In [1]:
from __future__ import annotations

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from IPython.display import display


## Official LargeST GLA structure

From the official LargeST repository:
- `data/ca/` is populated from the official Kaggle-hosted California dataset.
- `data/gla/` in the repo contains `generate_gla_dataset.ipynb`, which is used to derive the GLA subset from CA.
- `data/generate_data_for_training.py` then expects a raw GLA HDF5 file named like `gla/gla_his_2019.h5` and writes processed training artifacts such as `gla/2019/his.npz`, `idx_train.npy`, `idx_val.npy`, and `idx_test.npy`.

This means the official repo does not ship the GLA subset directly inside Git history; it documents how to derive it locally.


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


repo_root = find_repo_root(Path.cwd())

candidate_gla_roots = [
    repo_root / "data" / "raw" / "LargeST" / "gla",
    repo_root / "data" / "raw" / "LargeST-GLA",
    repo_root / "data" / "raw" / "largest-gla",
    repo_root / "data" / "raw" / "gla",
    repo_root / "data" / "gla",
]

gla_root = first_existing(candidate_gla_roots)
gla_root


PosixPath('/Users/jerry/Projects/School/intellipath-ml/data/raw/LargeST/gla')

In [3]:
def discover_candidate_files(base_dir: Path | None) -> dict[str, Path | None]:
    if base_dir is None:
        return {
            "raw_h5": None,
            "processed_npz": None,
            "metadata_csv": None,
            "adjacency": None,
        }

    files = list(base_dir.rglob("*"))

    def pick(matchers: list[str]) -> Path | None:
        for file in files:
            if not file.is_file():
                continue
            lower_name = file.name.lower()
            if all(token in lower_name for token in matchers):
                return file
        return None

    raw_h5 = pick(["gla", ".h5"]) or pick(["his", ".h5"])
    processed_npz = pick(["his", ".npz"])
    metadata_csv = pick(["meta", ".csv"]) or pick(["sensor", ".csv"])
    adjacency = (
        pick(["adj", ".npy"])
        or pick(["adj", ".npz"])
        or pick(["adj", ".csv"])
        or pick(["adj", ".pkl"])
        or pick(["graph", ".csv"])
    )

    return {
        "raw_h5": raw_h5,
        "processed_npz": processed_npz,
        "metadata_csv": metadata_csv,
        "adjacency": adjacency,
    }


discovered = discover_candidate_files(gla_root)
display(pd.DataFrame({"path": {k: str(v) if v else None for k, v in discovered.items()}}))


,path
raw_h5,/Users/jerry/Projects/School/intellipath-ml/da...
processed_npz,NaN
metadata_csv,/Users/jerry/Projects/School/intellipath-ml/da...
adjacency,/Users/jerry/Projects/School/intellipath-ml/da...


## Local inspection helpers

These functions load whichever artifacts are available locally. If only processed training bundles exist, the notebook still reports tensor shapes, but timestamp coverage may remain unavailable because the generated `.npz` files do not include the original datetime index.


In [4]:
def load_flow_hdf(path: Path | None) -> pd.DataFrame | None:
    if path is None or not path.exists():
        return None
    try:
        df = pd.read_hdf(path)
    except (ValueError, KeyError):
        with pd.HDFStore(path, mode="r") as store:
            keys = store.keys()
            if not keys:
                return None
            df = store[keys[0]]
    df.index = pd.to_datetime(df.index)
    return df.sort_index()


def load_processed_npz(path: Path | None) -> dict[str, np.ndarray | float] | None:
    if path is None or not path.exists():
        return None
    bundle = np.load(path)
    return {key: bundle[key] for key in bundle.files}


def load_metadata_csv(path: Path | None) -> pd.DataFrame | None:
    if path is None or not path.exists():
        return None
    return pd.read_csv(path)


def load_adjacency(path: Path | None):
    if path is None or not path.exists():
        return None

    suffix = path.suffix.lower()
    if suffix == ".npy":
        return np.load(path, allow_pickle=True)
    if suffix == ".npz":
        loaded = np.load(path, allow_pickle=True)
        first_key = loaded.files[0]
        return loaded[first_key]
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".pkl":
        with open(path, "rb") as file:
            return pickle.load(file)
    return None


def summarize_adjacency(adjacency_obj) -> tuple[dict[str, object], object | None]:
    if adjacency_obj is None:
        return {"type": None, "num_nodes": None, "num_edges": None}, None

    if isinstance(adjacency_obj, pd.DataFrame):
        numeric = adjacency_obj.select_dtypes(include=[np.number])
        if numeric.shape[0] == numeric.shape[1] and numeric.shape[0] > 0:
            matrix = numeric.to_numpy()
            edge_count = int(np.count_nonzero(matrix) - np.count_nonzero(np.diag(matrix)))
            preview = numeric.iloc[:5, :5]
            return {
                "type": "dataframe-matrix",
                "num_nodes": int(matrix.shape[0]),
                "num_edges": edge_count,
            }, preview
        return {
            "type": "dataframe-table",
            "num_nodes": None,
            "num_edges": int(len(adjacency_obj)),
        }, adjacency_obj.head()

    if isinstance(adjacency_obj, np.ndarray):
        if adjacency_obj.ndim == 2:
            edge_count = int(np.count_nonzero(adjacency_obj) - np.count_nonzero(np.diag(adjacency_obj)))
            preview = pd.DataFrame(adjacency_obj[:5, :5])
            return {
                "type": "ndarray-matrix",
                "num_nodes": int(adjacency_obj.shape[0]),
                "num_edges": edge_count,
            }, preview
        return {
            "type": "ndarray",
            "num_nodes": None,
            "num_edges": None,
        }, adjacency_obj[:5]

    if isinstance(adjacency_obj, tuple):
        tuple_summary = {
            "type": "tuple",
            "num_nodes": None,
            "num_edges": None,
        }
        try:
            adjacency_matrix = np.asarray(adjacency_obj[0])
            if adjacency_matrix.ndim == 2:
                tuple_summary["num_nodes"] = int(adjacency_matrix.shape[0])
                tuple_summary["num_edges"] = int(
                    np.count_nonzero(adjacency_matrix) - np.count_nonzero(np.diag(adjacency_matrix))
                )
                return tuple_summary, pd.DataFrame(adjacency_matrix[:5, :5])
        except Exception:
            pass
        return tuple_summary, adjacency_obj

    return {
        "type": type(adjacency_obj).__name__,
        "num_nodes": None,
        "num_edges": None,
    }, None


In [5]:
flow_df = load_flow_hdf(discovered["raw_h5"])
processed = load_processed_npz(discovered["processed_npz"])
metadata_df = load_metadata_csv(discovered["metadata_csv"])
adjacency_obj = load_adjacency(discovered["adjacency"])
adjacency_summary, adjacency_preview = summarize_adjacency(adjacency_obj)

if flow_df is not None:
    time_range = (str(flow_df.index.min()), str(flow_df.index.max()))
    flow_shape = flow_df.shape
    inferred_nodes = flow_df.shape[1]
else:
    time_range = None
    flow_shape = None
    inferred_nodes = None

if processed is not None:
    processed_shapes = {key: np.shape(value) for key, value in processed.items()}
    if inferred_nodes is None and "data" in processed and np.ndim(processed["data"]) >= 2:
        inferred_nodes = int(processed["data"].shape[1])
else:
    processed_shapes = None

if inferred_nodes is None and metadata_df is not None:
    inferred_nodes = int(len(metadata_df))
if inferred_nodes is None and adjacency_summary["num_nodes"] is not None:
    inferred_nodes = int(adjacency_summary["num_nodes"])

summary = pd.Series(
    {
        "gla_root": str(gla_root) if gla_root else None,
        "raw_h5_found": discovered["raw_h5"] is not None,
        "processed_npz_found": discovered["processed_npz"] is not None,
        "metadata_found": discovered["metadata_csv"] is not None,
        "adjacency_found": discovered["adjacency"] is not None,
        "raw_flow_shape": flow_shape,
        "processed_shapes": processed_shapes,
        "timestamp_range": time_range,
        "num_sensors_or_nodes": inferred_nodes,
        "num_edges": adjacency_summary["num_edges"],
        "adjacency_type": adjacency_summary["type"],
    }
)

display(summary.to_frame("value"))


,value
gla_root,/Users/jerry/Projects/School/intellipath-ml/da...
raw_h5_found,True
processed_npz_found,False
metadata_found,True
adjacency_found,True
raw_flow_shape,"(35040, 3834)"
processed_shapes,None
timestamp_range,"(2019-01-01 00:00:00, 2019-12-31 23:45:00)"
num_sensors_or_nodes,3834
num_edges,98703


In [6]:
if flow_df is not None:
    print("Raw flow preview")
    display(flow_df.head())
else:
    print("Raw flow preview unavailable: no local GLA HDF5 file detected.")

if metadata_df is not None:
    print("Metadata preview")
    display(metadata_df.head())
else:
    print("Metadata preview unavailable: no local metadata CSV detected.")

if adjacency_preview is not None:
    print("Adjacency / graph preview")
    display(adjacency_preview)
else:
    print("Adjacency preview unavailable: no local adjacency artifact detected.")


Raw flow preview


,767494,767509,767523,767541,767554,767572,767620,767455,768202,767470,...,1221523,1221550,1221536,1221556,1219551,1202527,1202549,1219560,1202522,1202537
Time,,,,,,,,,,,,,,,,,,,,,
2019-01-01 00:00:00,24.0,89.0,60.0,63.0,51.0,57.0,106.0,37.0,18.0,49.0,...,102.0,105.0,93.0,76.0,52.0,0.0,0.0,26.0,64.0,58.0
2019-01-01 00:15:00,72.0,173.0,158.0,171.0,140.0,142.0,122.0,73.0,60.0,127.0,...,178.0,166.0,121.0,98.0,137.0,127.0,0.0,56.0,122.0,118.0
2019-01-01 00:30:00,107.0,227.0,235.0,259.0,202.0,214.0,166.0,107.0,99.0,194.0,...,255.0,235.0,186.0,162.0,183.0,167.0,0.0,147.0,149.0,137.0
2019-01-01 00:45:00,97.0,219.0,227.0,250.0,203.0,210.0,173.0,102.0,96.0,196.0,...,270.0,268.0,217.0,180.0,186.0,168.0,0.0,169.0,157.0,154.0
2019-01-01 01:00:00,99.0,231.0,226.0,248.0,190.0,199.0,160.0,98.0,90.0,174.0,...,280.0,281.0,222.0,180.0,141.0,120.0,0.0,175.0,171.0,163.0


Metadata preview


,ID,Lat,Lng,District,County,Fwy,Lanes,Type,Direction,ID2
0,767494,34.103332,-118.249733,7,Los Angeles,SR2-E,3,Mainline,E,3527
1,767509,34.107693,-118.249538,7,Los Angeles,SR2-E,4,Mainline,E,3528
2,767523,34.113959,-118.242956,7,Los Angeles,SR2-E,4,Mainline,E,3529
3,767541,34.116060,-118.238384,7,Los Angeles,SR2-E,4,Mainline,E,3530
4,767554,34.121074,-118.229892,7,Los Angeles,SR2-E,4,Mainline,E,3531


Adjacency / graph preview


,0,1,2,3,4
0,1.000000,0.988261,0.899386,0.828356,0.654618
1,0.180545,1.000000,0.954013,0.899597,0.745245
2,0.050767,0.172985,1.000000,0.988338,0.899597
3,0.072929,0.227784,0.328125,1.000000,0.954013
4,0.000000,0.000000,0.012506,0.060366,1.000000


## Minimal local loading template

If the dataset is not present yet, this is the smallest reasonable project-side loader shape to start from later.


In [7]:
example_gla_root = repo_root / "data" / "raw" / "LargeST" / "gla"
example_raw_h5 = example_gla_root / "gla_his_2019.h5"
example_metadata_csv = example_gla_root / "gla_meta.csv"
example_adjacency_npy = example_gla_root / "gla_rn_adj.npy"
example_processed_npz = example_gla_root / "2019" / "his.npz"

print("Example expected paths")
print(example_raw_h5)
print(example_metadata_csv)
print(example_adjacency_npy)
print(example_processed_npz)


Example expected paths
/Users/jerry/Projects/School/intellipath-ml/data/raw/LargeST/gla/gla_his_2019.h5
/Users/jerry/Projects/School/intellipath-ml/data/raw/LargeST/gla/gla_meta.csv
/Users/jerry/Projects/School/intellipath-ml/data/raw/LargeST/gla/gla_adj.npy
/Users/jerry/Projects/School/intellipath-ml/data/raw/LargeST/gla/2019/his.npz


## METR-LA vs LargeST-GLA for this project

- Temporal coverage: our local METR-LA file covers only `2012-03-01` through `2012-06-27`. Official LargeST is built from California data spanning `2017` to `2021`, and the official workflow generates GLA subsets for chosen year slices such as `2019` or multi-year combinations.
- Number of sensors: METR-LA has `207` sensors locally. LargeST-GLA is a much larger regional graph derived from the `8,600`-sensor California corpus, so it should be substantially larger than METR-LA even though the exact local node count depends on the generated subset files.
- LA relevance: both datasets are Los Angeles-area traffic benchmarks, but LargeST-GLA is more aligned with a broader Greater LA story than the smaller METR-LA benchmark slice.
- Likely implementation complexity: METR-LA is easier because it is already present and compact. LargeST-GLA is more attractive if we want broader temporal coverage and a more realistic large-graph setting, but it adds data-prep work because the official repo derives GLA from the larger California dataset rather than shipping a tiny ready-to-use benchmark file in the repo.


## Feasibility takeaway

Switching looks feasible, but not drop-in. The main cost is data acquisition and subset generation, not model code. If we want to preserve momentum, the next low-risk step would be to download the official LargeST CA data, derive the GLA subset once, and then decide whether the extra coverage is worth the larger graph and preprocessing overhead.
